In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [ ]:
!git clone -b dream-finetune https://github.com/kainspraveen/marl-drug-discovery.git


In [ ]:
!pip install "transformers==4.57.1" peft bitsandbytes datasets trl accelerate qwen-vl-utils flash-attn wandb verl rdkit

In [ ]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=secret_value_0)

In [ ]:
import sys
sys.path.append("/kaggle/working/marl-drug-discovery")


In [ ]:

from DiffusionLM.src.sft_dataset import SFTDataset
from DiffusionLM.src.sft_trainer import FSDPSFTTrainer
config_yaml = "/kaggle/input/datasets/kainspraveen/config-yaml"

In [ ]:
import os

# Set the env vars that torchrun normally sets
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"

import torch
if not torch.distributed.is_initialized():
    torch.distributed.init_process_group(backend="nccl", world_size=1, rank=0)
    torch.cuda.set_device(0)

print(f"Distributed initialized: {torch.distributed.is_initialized()}")
print(f"World size: {torch.distributed.get_world_size()}")

In [ ]:
import zipfile
dataset_url = "https://huggingface.co/datasets/zjunlp/Mol-Instructions/resolve/main/data/Molecule-oriented_Instructions.zip"
zip_file = "Molecule-oriented_Instructions.zip"
extract_dir = "./data_mol_instruct"

# Download and unzip if checking fails
if not os.path.exists(extract_dir):
    print(f"Downloading {dataset_url}...")
    os.system(f"wget -q {dataset_url} -O {zip_file}")
    
    print(f"Extracting {zip_file}...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete.")
else:
    print("Dataset already downloaded and extracted.")


In [ ]:
#prepare data




import pandas as pd
import json
import os

# Load the JSON data
json_path = os.path.join("./", "data_mol_instruct/Molecule-oriented_Instructions/description_guided_molecule_design.json")
with open(json_path, 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

def format_prompt(row):
    instruction = row.get('instruction', '')
    inp = row.get('input', '')
    if inp:
        return f"{instruction}\nInput: {inp}"
    return instruction

df['prompt'] = df.apply(format_prompt, axis=1)
df['response'] = df['output']

# Split into train and validation (simple split for demonstration)

df['split'] = df['metadata'].apply(lambda m: m['split'])
train_df_ = df[df['split'] == 'train'][['prompt', 'response']]
train_df = train_df_.sample(frac=0.9, random_state=42)
val_df = train_df_.drop(train_df.index)
test_df = df[df['split'] == 'test'][['prompt', 'response']]


# Save to Parquet
train_parquet_path = os.path.join("./", "train.parquet")
val_parquet_path = os.path.join("./", "val.parquet")

# os.makedirs(OUTPUT_DIR, exist_ok=True)
train_df.to_parquet(train_parquet_path)
val_df.to_parquet(val_parquet_path)

print(f"Saved train parquet to {train_parquet_path}")
print(f"Saved val parquet to {val_parquet_path}")

# TRAIN_FILE = train_parquet_path
# VAL_FILE = val_parquet_path
print(df['split'].value_counts())
print(len(train_df), len(val_df), len(test_df))

In [ ]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from torch.distributed.device_mesh import init_device_mesh

# Point to the config directory (absolute path)
config_dir = "/kaggle/input/datasets/kainspraveen/config-yaml"
TRAIN_FILE = "./train.parquet"
VAL_FILE = "./val.parquet"
OUTPUT_DIR = "./output_dir"
with initialize_config_dir(config_dir=config_dir, version_base=None):
    config = compose(
        config_name="sft_trainer",
        overrides=[
            "diffusion.time_reweighting=cart",
            "diffusion.cart_p=0.3",
            f"data.train_files={TRAIN_FILE}",       # your train parquet path
            f"data.val_files={VAL_FILE}",             # your val parquet path
            "data.max_length=1024",
            "data.prompt_key=prompt",
            "data.response_key=response",
            "data.truncation=right",
            "optim.lr=2e-4",
            "data.micro_batch_size_per_gpu=4",
            "+data.enable_perbatch_cutoff=True",
            "data.perbatch_cutoff_type=random_with_input_pad",
            "++data.perbatch_cutoff=True",
            "++trainer.save_checkpoint_steps=250",
            "model.partial_pretrain=Dream-org/Dream-v0-Instruct-7B",
            "model.trust_remote_code=True",
            "model.enable_gradient_checkpointing=True",
            "model.lora_rank=512",
            "model.lora_alpha=256",
            f"trainer.default_local_dir={OUTPUT_DIR}",
            "trainer.project_name=Dream-7B-Instruct-Mol",
            "trainer.experiment_name=dream-sft",
            "trainer.logger=['console','wandb']",
            "trainer.total_epochs=3",
        ],
    )

print(OmegaConf.to_yaml(config))

# Create device meshes (single GPU)
device_mesh = init_device_mesh("cuda", mesh_shape=(1,), mesh_dim_names=("fsdp",))

dp_size = 1 // config.ulysses_sequence_parallel_size
ulysses_device_mesh = init_device_mesh(
    "cuda",
    mesh_shape=(dp_size, config.ulysses_sequence_parallel_size),
    mesh_dim_names=("dp", "sp"),
)

# Import and run the trainer
# from src.trainer.fsdp_sft_trainer import FSDPSFTTrainer

trainer = FSDPSFTTrainer(
    config=config,
    device_mesh=device_mesh,
    ulysses_device_mesh=ulysses_device_mesh,
)
trainer.fit()